In [ ]:
#κελί  0
import os
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns





### 1. Ρυθμίσεις & Προετοιμασία
Εισαγωγή των απαραίτητων βιβλιοθηκών και ορισμός των βασικών διαδρομών αρχείων για την ανάλυση.


In [ ]:
#κελί  1
# Config
DATA_DIR = ".\\clio"  # επειδη το εχω μέσα στον φάκελο clio
EVENTS_DIR = os.path.join(DATA_DIR, "events_data")
USERS_DIR  = os.path.join(DATA_DIR, "users_data")

MONTHS = ["2025-07", "2025-08", "2025-09", "2025-10"]  # July–Oct 2025

EVENT_FILES = [f"events_data_{m}.csv" for m in MONTHS]
USER_FILES  = [f"users_data_{m}.csv" for m in MONTHS]

TOURS_FILE = os.path.join(DATA_DIR, "tour_title_mapping.csv")
LANG_FILE  = os.path.join(DATA_DIR, "id_language_mapping.csv")

OUTPUT_FILE = os.path.join(DATA_DIR, "events_clean_july_oct_2025.csv")

### 2. Βοηθητικές Συναρτήσεις
Δημιουργία βοηθητικών συναρτήσεων για ασφαλή φόρτωση και ενοποίηση των μηνιαίων αρχείων δεδομένων.


In [ ]:
#κελί 2
# Helpers
def read_monthly_csv(folder: str, filenames: list[str]) -> pd.DataFrame:
    missing = [f for f in filenames if not os.path.exists(os.path.join(folder, f))]
    if missing:
        raise FileNotFoundError(
            f"Missing files in '{folder}':\n" + "\n".join(missing)
        )
    dfs = []
    for f in filenames:
        path = os.path.join(folder, f)

        # low_memory=False για να μη σπάει types σε chunks (σταματάει τα DtypeWarning)
        # dtype για audio columns ως string ώστε να καθαριστούν σωστά μετά (mixed types fix)
        df = pd.read_csv(
            path,
            low_memory=False,
            dtype={
                "audio_time_played": "string",
                "audio_time_paused": "string",
            }
        )

        df["source_file"] = f  # helpful for debugging
        dfs.append(df)

    return pd.concat(dfs, ignore_index=True)


### 3. Φόρτωση Δεδομένων (Extract)
Φόρτωση των δεδομένων γεγονότων, χρηστών και των αρχείων αντιστοίχισης, καθώς και ενοποίηση των μηνιαίων αρχείων.


In [ ]:
#κελί 3
# EXTRACT
events = read_monthly_csv(EVENTS_DIR, EVENT_FILES)
users  = read_monthly_csv(USERS_DIR, USER_FILES)

tours = pd.read_csv(TOURS_FILE)
languages = pd.read_csv(LANG_FILE)

print("Loaded:")
print("events:", events.shape)
print("users :", users.shape)
print("tours :", tours.shape)
print("languages:", languages.shape)

### 4. Επεξεργασία Ημερομηνιών & Χρονικών Σφραγίδων
Μετατροπή και τυποποίηση των πεδίων ημερομηνίας και χρονικών σφραγίδων για χρονική ανάλυση.


In [ ]:
# κελι 4
# TRANSFORM - dates/timestamps
# event_date robust parsing χωρίς .loc assigns
raw_date = events["event_date"].astype(str).str.strip()

mask_ymd8 = raw_date.str.fullmatch(r"\d{8}", na=False)

parsed_date = pd.Series(pd.NaT, index=events.index)

parsed_date.loc[mask_ymd8] = pd.to_datetime(
    raw_date.loc[mask_ymd8],
    format="%Y%m%d",
    errors="coerce"
)

parsed_date.loc[~mask_ymd8] = pd.to_datetime(
    raw_date.loc[~mask_ymd8],
    errors="coerce",
    dayfirst=True
)

events["event_date"] = parsed_date

ts_us = pd.to_datetime(events["event_timestamp"], unit="us", errors="coerce")
ts_ms = pd.to_datetime(events["event_timestamp"], unit="ms", errors="coerce")

def score_ts(ts):
    non_null = ts.notna().sum()
    year_ok = (ts.dt.year >= 2020).sum()
    return (year_ok, non_null)

events["event_timestamp"] = ts_us if score_ts(ts_us) >= score_ts(ts_ms) else ts_ms

# users timestamps
users["user_first_touch_timestamp_micros"] = pd.to_datetime(
    users["user_first_touch_timestamp_micros"], unit="us", errors="coerce"
)

if "first_purchase_date" in users.columns:
    users["first_purchase_date"] = pd.to_datetime(users["first_purchase_date"], errors="coerce")

# sanity checks
print("event_date dtype:", events["event_date"].dtype)
print("event_date null rate:", events["event_date"].isna().mean())
print("event_date min/max:", events["event_date"].min(), events["event_date"].max())
print("event_timestamp min/max:", events["event_timestamp"].min(), events["event_timestamp"].max())


### 5. Χρονικός Περιορισμός Δεδομένων
Φιλτράρισμα των δεδομένων ώστε να περιλαμβάνονται μόνο γεγονότα από τον Ιούλιο έως τον Οκτώβριο του 2025.


In [ ]:
# κελι 5
# TRANSFORM - filter July–Oct 2025
start_date = pd.Timestamp("2025-07-01")
end_date   = pd.Timestamp("2025-10-31")

before = len(events)

# Πρώτα πέτα NaT dates (αν υπάρχουν λίγα)
events = events.dropna(subset=["event_date"]).copy()

events = events[(events["event_date"] >= start_date) & (events["event_date"] <= end_date)].copy()

print("Rows before filter:", before)
print("Rows after  filter:", len(events))


### 6. Έλεγχος Ποιότητας Δεδομένων
Έλεγχος ελλιπών τιμών και αφαίρεση διπλότυπων εγγραφών για αξιόπιστη ανάλυση.


In [ ]:
#κελί 6
# TRANSFORM - missingness + duplicates
print("\nMissingness (top 10):")
print(events.isna().mean().sort_values(ascending=False).head(10))

# Αφαιρώ events που δεν έχουν βασικά IDs ή event_name/timestamp (συνήθως άχρηστα για analysis)
critical_cols = ["event_timestamp", "event_name"]
id_cols = ["user_id", "user_pseudo_id"]
key_cols = [c for c in (critical_cols + id_cols) if c in events.columns]

events_before = len(events)
events = events.dropna(subset=critical_cols)  # κρατάω events που έχουν τύπο & χρόνο
print(f"\nDropped {events_before - len(events)} rows with missing critical cols: {critical_cols}")

# Dedup με λογικό subset (όχι όλο το row) για να μη χάνεις χρήσιμα events από μικρές διαφορές
dedup_subset = [c for c in ["event_timestamp", "event_name", "user_id", "user_pseudo_id", "tour_id", "story_id"] if c in events.columns]

dups = events.duplicated(subset=dedup_subset).sum()
print(f"\nDuplicates (by {dedup_subset}): {dups}")

events = events.drop_duplicates(subset=dedup_subset).copy()

### 6.5 Καθαρισμός Audio Χρόνου
Μετατροπή των πεδίων audio_time_played και audio_time_paused σε αριθμητικές τιμές (δευτερόλεπτα), ώστε να είναι δυνατή η ανάλυση της διάρκειας ακρόασης.


In [ ]:
# cell 6.5
# Audio columns -> parse "mm:ss" / "hh:mm:ss" to seconds
for c in ["audio_time_played", "audio_time_paused"]:
    if c in events.columns:
        raw = events[c].astype("string").str.strip()

        # Most values are mm:ss (example: 00:45). Convert to 00:mm:ss.
        hhmmss = raw.where(raw.str.count(":") == 2, "00:" + raw)

        events[c + "_sec"] = pd.to_timedelta(hhmmss, errors="coerce").dt.total_seconds()

print(
    "audio_time_played raw non-null rate:",
    events["audio_time_played"].notna().mean() if "audio_time_played" in events.columns else "N/A",
)
print(
    "audio_time_paused raw non-null rate:",
    events["audio_time_paused"].notna().mean() if "audio_time_paused" in events.columns else "N/A",
)
print(
    "audio_time_played_sec non-null rate:",
    events["audio_time_played_sec"].notna().mean() if "audio_time_played_sec" in events.columns else "N/A",
)
print(
    "audio_time_paused_sec non-null rate:",
    events["audio_time_paused_sec"].notna().mean() if "audio_time_paused_sec" in events.columns else "N/A",
)


### 7. Εμπλουτισμός Δεδομένων
Σύνδεση των δεδομένων γεγονότων με πληροφορίες περιήγησης και γλώσσας για καλύτερη ερμηνεία.


In [ ]:
#κελί 7
# TRANSFORM - merges (safe)
# mapping tables έχουν μοναδικά keys πριν κάνεις merge
if "tour_id" in tours.columns:
    tours = tours.drop_duplicates(subset=["tour_id"])
if "lang_id" in languages.columns:
    languages = languages.drop_duplicates(subset=["lang_id"])

# Merge tours
if "tour_id" in events.columns and "tour_id" in tours.columns:
    events = events.merge(tours, on="tour_id", how="left")

# Merge languages
if "lang_id" in events.columns and "lang_id" in languages.columns:
    events = events.merge(languages, on="lang_id", how="left")


### 8. Εξερευνητική Ανάλυση (EDA)
Αρχική διερεύνηση της κατανομής των γεγονότων, της δραστηριότητας των χρηστών και των διαθέσιμων ξεναγήσεων.


In [ ]:
#κελί 8
# QUICK EDA (prints)
print("\nEvent types (top 20):")
print(events["event_name"].value_counts().head(20))

print("\nUnique users:", events["user_id"].nunique() if "user_id" in events.columns else "no user_id")
print("Unique tours:", events["tour_id"].nunique() if "tour_id" in events.columns else "no tour_id")

if "tour_title" in events.columns and "story_id" in events.columns:
    print("\nStories per tour (top 15):")
    print(events.groupby("tour_title")["story_id"].nunique().sort_values(ascending=False).head(15))

### 9. Αποθήκευση Καθαρισμένων Δεδομένων
Αποθήκευση του τελικού, καθαρισμένου συνόλου δεδομένων για περαιτέρω ανάλυση.


In [ ]:
#κελί 9
# LOAD - save cleaned dataset
events_clean = events.copy()
events_clean.to_csv(OUTPUT_FILE, index=False)

print(f"\nSaved cleaned events to: {OUTPUT_FILE}")
print("Final shape:", events_clean.shape)


## Final Analysis for the 3 Required Questions
This section answers the 3 required questions:
1. How deeply do users consume tour content?
2. Are users actively listening or passively letting audio play?
3. Do users follow intended story order or jump around?

Note: Drop-off is kept as an appendix diagnostic, not one of the 3 core questions.


In [ ]:
# cell 10 (memory-safe, self-contained)
# Keep only columns needed by Q1/Q2/Q3 to reduce memory pressure.
analysis_cols = [
    "user_id",
    "user_pseudo_id",
    "event_timestamp",
    "platform",
    "event_name",
    "tour_id",
    "story_id",
    "tour_title",
    "channel",
]

available_cols = None
if "events_clean" in globals():
    available_cols = [c for c in analysis_cols if c in events_clean.columns]
    analysis_events = events_clean[available_cols]
else:
    from pathlib import Path
    default_clean_path = Path("./clio/events_clean_july_oct_2025.csv")
    clean_path = Path(OUTPUT_FILE) if "OUTPUT_FILE" in globals() else default_clean_path
    sample_df = pd.read_csv(clean_path, nrows=1)
    available_cols = [c for c in analysis_cols if c in sample_df.columns]
    analysis_events = pd.read_csv(clean_path, usecols=available_cols, low_memory=False)

analysis_events["event_timestamp"] = pd.to_datetime(analysis_events["event_timestamp"], errors="coerce")
analysis_events["event_name"] = analysis_events["event_name"].astype("string")

# Keep only events used downstream (Q1/Q2/Q3).
analysis_event_names = {
    "start_tour",
    "story_start",
    "story_listened_20",
    "story_listened_40",
    "story_listened_60",
    "story_listened_80",
    "story_completed",
    "pause",
    "play",
    "forward_10",
    "backward_10",
    "next_story",
    "previous_story",
    "click_progress_bar",
    "click_story",
    "tour_item_clicked",
}

analysis_events = analysis_events.loc[
    analysis_events["event_name"].isin(analysis_event_names)
]

# Stable user key: prefer logged-in user_id, fallback to pseudo id
uid = pd.to_numeric(analysis_events["user_id"], errors="coerce").astype("Int64").astype("string")
pid = analysis_events["user_pseudo_id"].astype("string")

analysis_events["user_key"] = uid.radd("uid_")
mask_uid_missing = uid.isna()
analysis_events.loc[mask_uid_missing, "user_key"] = pid[mask_uid_missing].radd("pid_")

analysis_events["tour_id"] = pd.to_numeric(analysis_events["tour_id"], errors="coerce").astype("Int64")
analysis_events["story_id"] = pd.to_numeric(analysis_events["story_id"], errors="coerce").astype("Int64")
analysis_events["platform"] = analysis_events["platform"].astype("string").str.upper()

valid_mask = (
    analysis_events["event_name"].notna()
    & analysis_events["event_timestamp"].notna()
    & analysis_events["user_key"].notna()
)
analysis_events = analysis_events.loc[valid_mask]

# Drop raw IDs after user_key derivation to release memory.
analysis_events = analysis_events.drop(columns=["user_id", "user_pseudo_id"], errors="ignore")

# Compact dtypes for repeated strings.
for col in ["event_name", "platform", "channel"]:
    if col in analysis_events.columns:
        analysis_events[col] = analysis_events[col].astype("category")

analysis_events = analysis_events.reset_index(drop=True)

# Lightweight compatibility alias for downstream ad-hoc checks.
# Note: this is filtered to analysis events/columns, not the full raw table.
events_clean = analysis_events
import gc
gc.collect()

key_events = [
    "start_tour",
    "story_start",
    "story_listened_20",
    "story_listened_40",
    "story_listened_60",
    "story_listened_80",
    "story_completed",
]

instrumentation_check = (
    analysis_events[analysis_events["event_name"].isin(key_events)]
    .groupby(["platform", "event_name"], observed=True)["user_key"]
    .nunique()
    .unstack(fill_value=0)
)

print("Shape analysis_events:", analysis_events.shape)
instrumentation_check


### Data Quality Note
`story_start` is present on Android but not consistently logged on iOS.
For this reason:
- Q3 strict sequence uses Android + `story_start`.
- Q3 cross-platform proxy uses story-level events with `story_id` so iOS can be included.


In [ ]:
# cell 11 (tour-session journeys for Q1/Q2)
depth_event_to_pct = {
    "story_listened_20": 20,
    "story_listened_40": 40,
    "story_listened_60": 60,
    "story_listened_80": 80,
    "story_completed": 100,
}

story_evidence_events = ["story_start"] + list(depth_event_to_pct.keys())
strong_control_events = [
    "forward_10",
    "backward_10",
    "next_story",
    "previous_story",
    "click_progress_bar",
]

# Include start_tour so journeys with no story evidence are still counted in Q1/Q2.
q12_event_names = set(["start_tour"] + story_evidence_events + strong_control_events)
q12_cols = ["user_key", "tour_id", "story_id", "event_name", "event_timestamp", "platform", "tour_title"]

q12_events = (
    analysis_events[
        analysis_events["tour_id"].notna()
        & analysis_events["event_name"].isin(q12_event_names)
    ][q12_cols]
    .copy()
)

q12_events["tour_id"] = q12_events["tour_id"].astype("Int64")
q12_events["story_id"] = q12_events["story_id"].astype("Int64")
q12_events = q12_events.sort_values(["user_key", "tour_id", "event_timestamp"]).reset_index(drop=True)

# Journey split rule: new tour-journey after 30 minutes inactivity within same user+tour.
gap_min = (
    q12_events
    .groupby(["user_key", "tour_id"])["event_timestamp"]
    .diff()
    .dt.total_seconds()
    .div(60)
)
new_journey = gap_min.isna() | (gap_min > 30)
q12_events["journey_idx"] = (
    new_journey.groupby([q12_events["user_key"], q12_events["tour_id"]]).cumsum().astype("Int64")
)

journey_keys = ["user_key", "tour_id", "journey_idx"]

journey_platform = (
    q12_events
    .groupby(journey_keys, as_index=False)["platform"]
    .first()
    .rename(columns={"platform": "journey_platform"})
)

journey_base = (
    q12_events
    .groupby(journey_keys, as_index=False)
    .agg(first_event_ts=("event_timestamp", "min"))
)

controls_q12 = q12_events[q12_events["event_name"].isin(strong_control_events)]
journey_controls = (
    controls_q12
    .groupby(journey_keys, as_index=False)
    .size()
    .rename(columns={"size": "strong_control_events"})
)

story_events_q12 = q12_events[
    q12_events["event_name"].isin(story_evidence_events)
    & q12_events["story_id"].notna()
].copy()
story_events_q12["depth_pct"] = story_events_q12["event_name"].map(depth_event_to_pct).fillna(0).astype(int)

story_progress_journey = (
    story_events_q12
    .groupby(journey_keys + ["story_id"], as_index=False)
    .agg(
        max_depth=("depth_pct", "max"),
        first_story_ts=("event_timestamp", "min"),
    )
)

journey_depth_from_story = (
    story_progress_journey
    .groupby(journey_keys, as_index=False)
    .agg(
        max_depth=("max_depth", "max"),
        stories_touched=("story_id", "nunique"),
        stories_completed=("max_depth", lambda s: int((s == 100).sum())),
    )
)

# Keep journeys even when no story evidence exists.
journey_depth = journey_base.merge(journey_depth_from_story, on=journey_keys, how="left")
journey_depth["max_depth"] = journey_depth["max_depth"].fillna(0).astype(int)
journey_depth["stories_touched"] = journey_depth["stories_touched"].fillna(0).astype(int)
journey_depth["stories_completed"] = journey_depth["stories_completed"].fillna(0).astype(int)
journey_depth["completion_share_pct"] = np.where(
    journey_depth["stories_touched"] > 0,
    (journey_depth["stories_completed"] / journey_depth["stories_touched"] * 100).round(2),
    0.0,
)

# Canonical story order per tour from median position across sessions.
session_sequences_q12 = (
    story_progress_journey
    .sort_values(journey_keys + ["first_story_ts"])
    .groupby(journey_keys, as_index=False)["story_id"]
    .agg(list)
    .rename(columns={"story_id": "story_seq"})
)

seq_exploded_q12 = session_sequences_q12.explode("story_seq").rename(columns={"story_seq": "story_id"})
seq_exploded_q12["position"] = seq_exploded_q12.groupby(journey_keys).cumcount() + 1

canonical_positions_q12 = (
    seq_exploded_q12
    .groupby(["tour_id", "story_id"], as_index=False)["position"]
    .median()
    .sort_values(["tour_id", "position", "story_id"])
)
canonical_positions_q12["canonical_rank"] = canonical_positions_q12.groupby("tour_id").cumcount() + 1

final_rank_by_tour = (
    canonical_positions_q12
    .groupby("tour_id", as_index=False)["canonical_rank"]
    .max()
    .rename(columns={"canonical_rank": "final_canonical_rank"})
)

story_progress_journey = story_progress_journey.merge(
    canonical_positions_q12[["tour_id", "story_id", "canonical_rank"]],
    on=["tour_id", "story_id"],
    how="left",
)

story_progress_journey["canonical_rank"] = story_progress_journey["canonical_rank"].fillna(0).astype(int)
story_progress_journey["canonical_rank_endlike"] = np.where(
    story_progress_journey["max_depth"] >= 80,
    story_progress_journey["canonical_rank"],
    0,
)

journey_rank_progress = (
    story_progress_journey
    .groupby(journey_keys, as_index=False)
    .agg(
        max_canonical_rank_seen=("canonical_rank", "max"),
        max_canonical_rank_endlike=("canonical_rank_endlike", "max"),
    )
)

journey_progress = (
    journey_depth
    .merge(journey_platform, on=journey_keys, how="left")
    .merge(journey_controls, on=journey_keys, how="left")
    .merge(journey_rank_progress, on=journey_keys, how="left")
    .merge(final_rank_by_tour, on="tour_id", how="left")
)

journey_progress["strong_control_events"] = journey_progress["strong_control_events"].fillna(0).astype(int)
journey_progress["max_canonical_rank_seen"] = journey_progress["max_canonical_rank_seen"].fillna(0).astype(int)
journey_progress["max_canonical_rank_endlike"] = journey_progress["max_canonical_rank_endlike"].fillna(0).astype(int)
journey_progress["final_canonical_rank"] = journey_progress["final_canonical_rank"].fillna(np.inf)

journey_progress["reached_last_story"] = (
    journey_progress["max_canonical_rank_seen"] >= journey_progress["final_canonical_rank"]
)
journey_progress["reached_tour_end"] = (
    journey_progress["max_canonical_rank_endlike"] >= journey_progress["final_canonical_rank"]
)

journey_progress["controls_per_story"] = np.where(
    journey_progress["stories_touched"] > 0,
    journey_progress["strong_control_events"] / journey_progress["stories_touched"],
    0,
)
journey_progress["controls_per_story"] = journey_progress["controls_per_story"].round(3)

# Active rule: >=2 strong controls OR >=0.2 controls/story.
journey_progress["listening_mode"] = np.where(
    (journey_progress["strong_control_events"] >= 2)
    | (journey_progress["controls_per_story"] >= 0.2),
    "Active listening",
    "Passive play",
)
journey_progress["journey_status"] = np.where(
    journey_progress["reached_tour_end"],
    "Reached tour end",
    "Abandoned before end",
)

print("Tour journeys (30-min sessions):", len(journey_progress))
journey_progress.head()



### Q1: How many users reach the end of a tour vs abandon it?
Definition used in this analysis (tour-session level):
- Journey key: `user_key + tour_id + journey_idx` where `journey_idx` splits on 30-minute inactivity gaps.
- Canonical order is inferred per tour from median story position across sessions.
- `Reached tour end`: journey reached the last canonical story with depth `>=80%` (or `story_completed`).
- `Abandoned before end`: journey did not reach the last canonical story with end-like depth.


In [ ]:
# cell 12 (Q1)
q1_journey_dist = (
    journey_progress["journey_status"]
    .value_counts()
    .rename_axis("journey_status")
    .reset_index(name="tour_journeys")
)
q1_journey_dist["share_pct"] = (
    q1_journey_dist["tour_journeys"] / q1_journey_dist["tour_journeys"].sum() * 100
).round(2)

q1_user_status = (
    journey_progress
    .groupby("user_key", as_index=False)
    .agg(
        journeys=("tour_id", "size"),
        any_reached_end=("reached_tour_end", "any"),
    )
)
q1_user_status["user_status"] = np.where(
    q1_user_status["any_reached_end"],
    "Reached end (at least once)",
    "Only abandoned",
)

q1_user_dist = (
    q1_user_status["user_status"]
    .value_counts()
    .rename_axis("user_status")
    .reset_index(name="users")
)
q1_user_dist["share_pct"] = (q1_user_dist["users"] / q1_user_dist["users"].sum() * 100).round(2)

q1_total_users = int(len(q1_user_status))
q1_reached_users = int(q1_user_status["any_reached_end"].sum())
q1_abandoned_only_users = q1_total_users - q1_reached_users

print("Q1 journey outcomes:")
print(q1_journey_dist)
print("")
print("Q1 user outcomes:")
print(q1_user_dist)

q1_journey_dist


In [ ]:
# cell 13 (Q1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.barplot(
    data=q1_journey_dist,
    x="journey_status",
    y="share_pct",
    ax=axes[0],
    color="#1f77b4",
)
axes[0].set_title("Q1: Journey outcomes")
axes[0].set_xlabel("Journey status")
axes[0].set_ylabel("Share of tour journeys (%)")
axes[0].tick_params(axis="x", rotation=20)

sns.barplot(
    data=q1_user_dist,
    x="user_status",
    y="share_pct",
    ax=axes[1],
    color="#2ca02c",
)
axes[1].set_title("Q1: User outcomes")
axes[1].set_xlabel("User status")
axes[1].set_ylabel("Share of users (%)")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

q1_user_status["journeys"].describe(percentiles=[0.25, 0.5, 0.75]).round(2)


### Q2: How many users are active vs passive inside a tour?
Definition used in this analysis (same tour-session journeys as Q1):
- `Active listening`: journey has `>=2` strong controls OR `controls_per_story >= 0.2`.
- `Passive play`: journey does not meet the active rule.

Strong control events: `forward_10`, `backward_10`, `next_story`, `previous_story`, `click_progress_bar`.


In [ ]:
# cell 14a (Q2)
q2_mode_dist = (
    journey_progress["listening_mode"]
    .value_counts()
    .rename_axis("listening_mode")
    .reset_index(name="tour_journeys")
)
q2_mode_dist["share_pct"] = (
    q2_mode_dist["tour_journeys"] / q2_mode_dist["tour_journeys"].sum() * 100
).round(2)

q2_by_platform = (
    journey_progress
    .groupby(["journey_platform", "listening_mode"], as_index=False, observed=True)
    .size()
    .rename(columns={"size": "tour_journeys"})
)
q2_by_platform["share_pct"] = (
    q2_by_platform["tour_journeys"]
    / q2_by_platform.groupby("journey_platform", observed=True)["tour_journeys"].transform("sum")
    * 100
).round(2)

q2_completion = (
    journey_progress
    .groupby("listening_mode", as_index=False)
    .agg(
        reach_end_rate_pct=("reached_tour_end", lambda s: round(s.mean() * 100, 2)),
        avg_completion_share_pct=("completion_share_pct", lambda s: round(s.mean(), 2)),
        avg_controls_per_story=("controls_per_story", lambda s: round(s.mean(), 3)),
        tour_journeys=("tour_id", "size"),
    )
)

q2_user_mode = (
    journey_progress
    .groupby("user_key", as_index=False)
    .agg(
        any_active=("listening_mode", lambda s: (s == "Active listening").any()),
        any_passive=("listening_mode", lambda s: (s == "Passive play").any()),
    )
)
q2_user_mode["user_mode"] = np.select(
    [
        q2_user_mode["any_active"] & q2_user_mode["any_passive"],
        q2_user_mode["any_active"],
        q2_user_mode["any_passive"],
    ],
    ["Mixed (active + passive)", "Active only", "Passive only"],
    default="Unknown",
)

q2_user_dist = (
    q2_user_mode["user_mode"]
    .value_counts()
    .rename_axis("user_mode")
    .reset_index(name="users")
)
q2_user_dist["share_pct"] = (q2_user_dist["users"] / q2_user_dist["users"].sum() * 100).round(2)

q2_total_users = int(len(q2_user_mode))
q2_active_users = int(q2_user_mode["any_active"].sum())
q2_passive_only_users = int((~q2_user_mode["any_active"]).sum())
q2_active_only_users = int((q2_user_mode["any_active"] & ~q2_user_mode["any_passive"]).sum())
q2_mixed_users = int((q2_user_mode["any_active"] & q2_user_mode["any_passive"]).sum())

print("Q2 distribution (journeys):")
print(q2_mode_dist)
print("")
print("Q2 user distribution:")
print(q2_user_dist)
print("")
print("Q2 reach-end rate by mode:")
print(q2_completion)



In [ ]:
# cell 14b (Q2)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.barplot(
    data=q2_mode_dist,
    x="listening_mode",
    y="share_pct",
    ax=axes[0],
    color="#4c78a8",
)
axes[0].set_title("Q2: Active vs Passive (journeys)")
axes[0].set_xlabel("Listening mode")
axes[0].set_ylabel("Share of tour journeys (%)")
axes[0].tick_params(axis="x", rotation=20)

sns.barplot(
    data=q2_user_dist,
    x="user_mode",
    y="share_pct",
    ax=axes[1],
    color="#72b7b2",
)
axes[1].set_title("Q2: User activity profiles")
axes[1].set_xlabel("User mode")
axes[1].set_ylabel("Share of users (%)")
axes[1].tick_params(axis="x", rotation=20)

sns.barplot(
    data=q2_completion,
    x="listening_mode",
    y="reach_end_rate_pct",
    ax=axes[2],
    color="#f58518",
)
axes[2].set_title("Q2: Reached end rate by mode")
axes[2].set_xlabel("Listening mode")
axes[2].set_ylabel("Reached tour end (%)")
axes[2].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()


In [ ]:
# cell 14c (Q2 summary)
def q2_value_or_zero(df, key_col, key_val, value_col):
    vals = df.loc[df[key_col] == key_val, value_col]
    return float(vals.iloc[0]) if len(vals) else 0.0


def q2_count_or_zero(df, key_col, key_val, value_col):
    vals = df.loc[df[key_col] == key_val, value_col]
    return int(vals.iloc[0]) if len(vals) else 0


q2_active_share = q2_value_or_zero(q2_mode_dist, "listening_mode", "Active listening", "share_pct")
q2_passive_share = q2_value_or_zero(q2_mode_dist, "listening_mode", "Passive play", "share_pct")
q2_active_journeys = q2_count_or_zero(q2_mode_dist, "listening_mode", "Active listening", "tour_journeys")
q2_passive_journeys = q2_count_or_zero(q2_mode_dist, "listening_mode", "Passive play", "tour_journeys")

q2_summary = pd.DataFrame(
    {
        "metric": [
            "Q2 - Active journeys (count)",
            "Q2 - Passive journeys (count)",
            "Q2 - Active journey share (%)",
            "Q2 - Passive journey share (%)",
            "Q2 - Users with >=1 active journey (count)",
            "Q2 - Users with >=1 active journey (%)",
            "Q2 - Users passive-only (count)",
            "Q2 - Users passive-only (%)",
            "Q2 - Reached-end rate (active journeys, %)",
            "Q2 - Reached-end rate (passive journeys, %)",
            "Q2 - Avg controls per story (active journeys)",
            "Q2 - Avg controls per story (passive journeys)",
        ],
        "value": [
            q2_active_journeys,
            q2_passive_journeys,
            q2_active_share,
            q2_passive_share,
            q2_active_users,
            round(q2_active_users / q2_total_users * 100, 2),
            q2_passive_only_users,
            round(q2_passive_only_users / q2_total_users * 100, 2),
            q2_value_or_zero(q2_completion, "listening_mode", "Active listening", "reach_end_rate_pct"),
            q2_value_or_zero(q2_completion, "listening_mode", "Passive play", "reach_end_rate_pct"),
            q2_value_or_zero(q2_completion, "listening_mode", "Active listening", "avg_controls_per_story"),
            q2_value_or_zero(q2_completion, "listening_mode", "Passive play", "avg_controls_per_story"),
        ],
    }
)

q2_summary



### Q3: Do users follow the intended story order or jump around?
Because we do not have explicit ground-truth sequence in the mapping table, we estimate a common order per tour from observed behavior and check whether each session follows it. This analysis is restricted to Android where `story_start` is present.

Event-based jump indicator: `previous_story`, `next_story`, `click_story`, `tour_item_clicked`.


In [ ]:
# cell 16 (Q3)
android_q3_events = (
    analysis_events[
        (analysis_events["platform"] == "ANDROID")
        & analysis_events["tour_id"].notna()
    ][["user_key", "tour_id", "tour_title", "story_id", "event_name", "event_timestamp"]]
    .copy()
)

android_q3_events["tour_id"] = android_q3_events["tour_id"].astype("Int64")
android_q3_events["story_id"] = android_q3_events["story_id"].astype("Int64")
android_q3_events = android_q3_events.sort_values(["user_key", "tour_id", "event_timestamp"]).reset_index(drop=True)

# Sessionization: split journeys after 30 minutes inactivity per user+tour.
android_gap_min = (
    android_q3_events
    .groupby(["user_key", "tour_id"])["event_timestamp"]
    .diff()
    .dt.total_seconds()
    .div(60)
)
android_q3_events["session_idx"] = (
    (android_gap_min.isna() | (android_gap_min > 30))
    .groupby([android_q3_events["user_key"], android_q3_events["tour_id"]])
    .cumsum()
    .astype("Int64")
)

android_starts = android_q3_events[
    (android_q3_events["event_name"] == "story_start")
    & android_q3_events["story_id"].notna()
].copy()


def unique_in_order(values):
    seen = set()
    out = []
    for value in values:
        if pd.isna(value):
            continue
        value = int(value)
        if value not in seen:
            seen.add(value)
            out.append(value)
    return out


def first_non_null(values):
    values = values.dropna()
    return values.iloc[0] if len(values) else pd.NA


session_sequences = (
    android_starts
    .groupby(["user_key", "tour_id", "session_idx"], dropna=False)["story_id"]
    .apply(unique_in_order)
    .reset_index(name="story_seq")
)

session_meta = (
    android_starts
    .groupby(["user_key", "tour_id", "session_idx"], as_index=False)
    .agg(
        tour_title=("tour_title", first_non_null),
        session_start_ts=("event_timestamp", "min"),
    )
)

session_sequences = session_sequences.merge(
    session_meta,
    on=["user_key", "tour_id", "session_idx"],
    how="left",
)

session_sequences["n_unique_stories"] = session_sequences["story_seq"].str.len()
session_sequences = session_sequences[session_sequences["n_unique_stories"] >= 2].reset_index(drop=True)
session_sequences["session_id"] = session_sequences.index

print("Android sessions with >=2 unique stories:", len(session_sequences))
session_sequences.head()



In [ ]:
# cell 17 (Q3)
seq_exploded = session_sequences[["session_id", "tour_id", "story_seq"]].explode("story_seq")
seq_exploded["position"] = seq_exploded.groupby("session_id").cumcount() + 1
seq_exploded = seq_exploded.rename(columns={"story_seq": "story_id"})

canonical_positions = (
    seq_exploded
    .groupby(["tour_id", "story_id"], as_index=False)["position"]
    .median()
    .sort_values(["tour_id", "position", "story_id"])
)
canonical_positions["canonical_rank"] = canonical_positions.groupby("tour_id").cumcount() + 1

rank_map = {
    (int(row.tour_id), int(row.story_id)): int(row.canonical_rank)
    for row in canonical_positions.itertuples(index=False)
}


def map_to_ranks(tour_id, seq):
    tid = int(tour_id)
    return [rank_map[(tid, int(s))] for s in seq if (tid, int(s)) in rank_map]


def is_monotonic_increasing(values):
    return all(b > a for a, b in zip(values, values[1:]))


session_sequences["canonical_rank_seq"] = session_sequences.apply(
    lambda row: map_to_ranks(row["tour_id"], row["story_seq"]),
    axis=1,
)
session_sequences["follows_common_order"] = session_sequences["canonical_rank_seq"].apply(
    is_monotonic_increasing
)

order_summary = (
    session_sequences["follows_common_order"]
    .value_counts()
    .rename_axis("follows_common_order")
    .reset_index(name="sessions")
)
order_summary["share_pct"] = (
    order_summary["sessions"] / order_summary["sessions"].sum() * 100
).round(2)

order_summary



In [ ]:
# cell 18 (Q3)
# Event-based jump indicator (secondary metric) at session level.
jump_events = ["previous_story", "next_story", "click_story", "tour_item_clicked"]

jump_sessions = (
    android_q3_events[
        android_q3_events["event_name"].isin(jump_events)
    ][["user_key", "tour_id", "session_idx"]]
    .drop_duplicates()
    .assign(has_jump_event=True)
)

session_sequences = session_sequences.merge(
    jump_sessions,
    on=["user_key", "tour_id", "session_idx"],
    how="left",
)

session_sequences["has_jump_event"] = session_sequences["has_jump_event"].eq(True)

jump_summary = (
    session_sequences["has_jump_event"]
    .value_counts()
    .rename_axis("has_jump_event")
    .reset_index(name="sessions")
)
jump_summary["share_pct"] = (
    jump_summary["sessions"] / jump_summary["sessions"].sum() * 100
).round(2)

jump_summary



In [ ]:
# cell 19 (Q3)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sns.barplot(
    data=order_summary,
    x="follows_common_order",
    y="share_pct",
    ax=axes[0],
    color="#ff7f0e",
)
axes[0].set_title("Q3: Common order vs jumping (Android)")
axes[0].set_xlabel("Follows common order")
axes[0].set_ylabel("Session share (%)")

sns.barplot(
    data=jump_summary,
    x="has_jump_event",
    y="share_pct",
    ax=axes[1],
    color="#d62728",
)
axes[1].set_title("Q3: Use of jump navigation events")
axes[1].set_xlabel("Has jump events")
axes[1].set_ylabel("Session share (%)")

plt.tight_layout()
plt.show()


### Q3 (Cross-Platform Proxy): Include iOS without `story_start`
To include iOS, we infer story entry from story-level events that carry `story_id`.
Proxy story-entry events:
`play`, `pause`, `forward_10`, `backward_10`, `previous_story`, `click_progress_bar`, `story_listened_20/40/60/80`, `story_completed`.

Notes:
- This is a proxy sequence metric, not a strict `story_start` metric.
- We keep only sessions with at least 2 unique stories (`n_unique_stories >= 2`).

Event-based jump indicator: `previous_story`, `next_story`, `click_story`, `tour_item_clicked`.


In [ ]:
# cell 20a (Q3 cross-platform proxy)
proxy_entry_events = [
    "story_start",
    "play",
    "pause",
    "forward_10",
    "backward_10",
    "previous_story",
    "next_story",
    "click_progress_bar",
    "click_story",
    "tour_item_clicked",
    "story_listened_20",
    "story_listened_40",
    "story_listened_60",
    "story_listened_80",
    "story_completed",
]

proxy_story_events = (
    analysis_events[
        analysis_events["platform"].isin(["ANDROID", "IOS"])
        & analysis_events["event_name"].isin(proxy_entry_events)
        & analysis_events["tour_id"].notna()
        & analysis_events["story_id"].notna()
    ][["user_key", "tour_id", "tour_title", "story_id", "event_name", "event_timestamp", "platform"]]
    .copy()
)

proxy_story_events["tour_id"] = proxy_story_events["tour_id"].astype("Int64")
proxy_story_events["story_id"] = proxy_story_events["story_id"].astype("Int64")
proxy_story_events = proxy_story_events.sort_values(["user_key", "tour_id", "event_timestamp"]).reset_index(drop=True)

proxy_gap_min = (
    proxy_story_events
    .groupby(["user_key", "tour_id"])["event_timestamp"]
    .diff()
    .dt.total_seconds()
    .div(60)
)
proxy_story_events["session_idx"] = (
    (proxy_gap_min.isna() | (proxy_gap_min > 30))
    .groupby([proxy_story_events["user_key"], proxy_story_events["tour_id"]])
    .cumsum()
    .astype("Int64")
)

session_sequences_proxy = (
    proxy_story_events
    .groupby(["user_key", "tour_id", "session_idx"], dropna=False)["story_id"]
    .apply(unique_in_order)
    .reset_index(name="story_seq")
)

session_meta_proxy = (
    proxy_story_events
    .groupby(["user_key", "tour_id", "session_idx"], as_index=False)
    .agg(
        tour_title=("tour_title", first_non_null),
        journey_platform=("platform", "first"),
        session_start_ts=("event_timestamp", "min"),
    )
)

session_sequences_proxy = session_sequences_proxy.merge(
    session_meta_proxy,
    on=["user_key", "tour_id", "session_idx"],
    how="left",
)

session_sequences_proxy["n_unique_stories"] = session_sequences_proxy["story_seq"].str.len()
session_sequences_proxy = session_sequences_proxy[
    session_sequences_proxy["n_unique_stories"] >= 2
].reset_index(drop=True)
session_sequences_proxy["session_id"] = session_sequences_proxy.index

seq_exploded_proxy = session_sequences_proxy[["session_id", "tour_id", "story_seq"]].explode("story_seq")
seq_exploded_proxy["position"] = seq_exploded_proxy.groupby("session_id").cumcount() + 1
seq_exploded_proxy = seq_exploded_proxy.rename(columns={"story_seq": "story_id"})

canonical_positions_proxy = (
    seq_exploded_proxy
    .groupby(["tour_id", "story_id"], as_index=False)["position"]
    .median()
    .sort_values(["tour_id", "position", "story_id"])
)
canonical_positions_proxy["canonical_rank"] = canonical_positions_proxy.groupby("tour_id").cumcount() + 1

rank_map_proxy = {
    (int(row.tour_id), int(row.story_id)): int(row.canonical_rank)
    for row in canonical_positions_proxy.itertuples(index=False)
}


def map_to_ranks_proxy(tour_id, seq):
    tid = int(tour_id)
    return [rank_map_proxy[(tid, int(s))] for s in seq if (tid, int(s)) in rank_map_proxy]


session_sequences_proxy["canonical_rank_seq"] = session_sequences_proxy.apply(
    lambda row: map_to_ranks_proxy(row["tour_id"], row["story_seq"]),
    axis=1,
)
session_sequences_proxy["follows_common_order_proxy"] = session_sequences_proxy[
    "canonical_rank_seq"
].apply(is_monotonic_increasing)

order_summary_proxy = (
    session_sequences_proxy["follows_common_order_proxy"]
    .value_counts()
    .rename_axis("follows_common_order_proxy")
    .reset_index(name="sessions")
)
order_summary_proxy["share_pct"] = (
    order_summary_proxy["sessions"] / order_summary_proxy["sessions"].sum() * 100
).round(2)

order_summary_proxy_by_platform = (
    session_sequences_proxy
    .groupby(["journey_platform", "follows_common_order_proxy"], as_index=False)
    .size()
    .rename(columns={"size": "sessions"})
)
order_summary_proxy_by_platform["share_pct"] = (
    order_summary_proxy_by_platform["sessions"]
    / order_summary_proxy_by_platform.groupby("journey_platform")["sessions"].transform("sum")
    * 100
).round(2)

print("Cross-platform proxy sessions (>=2 stories):", len(session_sequences_proxy))
print("Sessions by platform:")
print(session_sequences_proxy["journey_platform"].value_counts())
order_summary_proxy



In [ ]:
# cell 20b (Q3 cross-platform proxy)
# Event-based jump indicator (secondary metric) at session level.
jump_events_proxy = ["previous_story", "next_story", "click_story", "tour_item_clicked"]

jump_sessions_proxy = (
    proxy_story_events[
        proxy_story_events["event_name"].isin(jump_events_proxy)
    ][["user_key", "tour_id", "session_idx"]]
    .drop_duplicates()
    .assign(has_jump_event_proxy=True)
)

session_sequences_proxy = session_sequences_proxy.merge(
    jump_sessions_proxy,
    on=["user_key", "tour_id", "session_idx"],
    how="left",
)
session_sequences_proxy["has_jump_event_proxy"] = session_sequences_proxy[
    "has_jump_event_proxy"
].eq(True)

jump_summary_proxy = (
    session_sequences_proxy["has_jump_event_proxy"]
    .value_counts()
    .rename_axis("has_jump_event_proxy")
    .reset_index(name="sessions")
)
jump_summary_proxy["share_pct"] = (
    jump_summary_proxy["sessions"] / jump_summary_proxy["sessions"].sum() * 100
).round(2)

jump_summary_proxy_by_platform = (
    session_sequences_proxy
    .groupby(["journey_platform", "has_jump_event_proxy"], as_index=False)
    .size()
    .rename(columns={"size": "sessions"})
)
jump_summary_proxy_by_platform["share_pct"] = (
    jump_summary_proxy_by_platform["sessions"]
    / jump_summary_proxy_by_platform.groupby("journey_platform")["sessions"].transform("sum")
    * 100
).round(2)


def share_or_zero(df, key_col, key_value):
    vals = df.loc[df[key_col] == key_value, "share_pct"]
    return float(vals.iloc[0]) if len(vals) else 0.0


def count_or_zero(df, key_col, key_value):
    vals = df.loc[df[key_col] == key_value, "sessions"]
    return int(vals.iloc[0]) if len(vals) else 0


q3_compare = pd.DataFrame(
    {
        "metric": [
            "Follows common order (sessions, count)",
            "Follows common order (%)",
            "Has jump event (sessions, count)",
            "Has jump event (%)",
            "Sessions analyzed",
        ],
        "Q3 strict (Android + story_start)": [
            count_or_zero(order_summary, "follows_common_order", True),
            share_or_zero(order_summary, "follows_common_order", True),
            count_or_zero(jump_summary, "has_jump_event", True),
            share_or_zero(jump_summary, "has_jump_event", True),
            int(len(session_sequences)),
        ],
        "Q3 proxy (Android + iOS)": [
            count_or_zero(order_summary_proxy, "follows_common_order_proxy", True),
            share_or_zero(order_summary_proxy, "follows_common_order_proxy", True),
            count_or_zero(jump_summary_proxy, "has_jump_event_proxy", True),
            share_or_zero(jump_summary_proxy, "has_jump_event_proxy", True),
            int(len(session_sequences_proxy)),
        ],
    }
)

q3_compare



In [ ]:
# cell 20c (Q3 cross-platform proxy)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sns.barplot(
    data=order_summary_proxy,
    x="follows_common_order_proxy",
    y="share_pct",
    ax=axes[0],
    color="#1f77b4",
)
axes[0].set_title("Q3 proxy: Common order vs jumping (cross-platform)")
axes[0].set_xlabel("Follows common order (proxy)")
axes[0].set_ylabel("Session share (%)")

sns.barplot(
    data=jump_summary_proxy,
    x="has_jump_event_proxy",
    y="share_pct",
    ax=axes[1],
    color="#d62728",
)
axes[1].set_title("Q3 proxy: Use of jump navigation events")
axes[1].set_xlabel("Has jump events (proxy)")
axes[1].set_ylabel("Session share (%)")

plt.tight_layout()
plt.show()


### Appendix A: Drop-off Diagnostic (Extra)
This is kept as supplementary analysis and is not one of the 3 required questions.


In [ ]:
# cell A1 (Appendix drop-off)
funnel_thresholds = [
    ("Started journey (>=0%)", 0),
    ("Reached 20%", 20),
    ("Reached 40%", 40),
    ("Reached 60%", 60),
    ("Reached 80%", 80),
    ("Completed (100%)", 100),
]

funnel_rows = []
for stage, threshold in funnel_thresholds:
    if threshold == 100:
        count = (journey_progress["max_depth"] == 100).sum()
    else:
        count = (journey_progress["max_depth"] >= threshold).sum()

    funnel_rows.append({"stage": stage, "tour_journeys": int(count)})

funnel_df = pd.DataFrame(funnel_rows)
funnel_df["dropoff_from_prev_pct"] = (
    1 - funnel_df["tour_journeys"] / funnel_df["tour_journeys"].shift(1)
).mul(100).round(2)
funnel_df.loc[0, "dropoff_from_prev_pct"] = 0.0

funnel_df


In [ ]:
# cell A2 (Appendix drop-off)
plt.figure(figsize=(9, 5))
sns.barplot(data=funnel_df, x="stage", y="tour_journeys", color="#2ca02c")
plt.title("Appendix: Drop-off by listening stage")
plt.xlabel("Stage")
plt.ylabel("Tour journeys")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

dropoff_peak = funnel_df.iloc[1:].sort_values("dropoff_from_prev_pct", ascending=False).head(1)
dropoff_peak


In [ ]:
# cell 20 (final summary for 3 required questions)
def summary_pct_or_zero(df, key_col, key_value, pct_col="share_pct"):
    vals = df.loc[df[key_col] == key_value, pct_col]
    return float(vals.iloc[0]) if len(vals) else 0.0


def summary_count_or_zero(df, key_col, key_value, count_col="sessions"):
    vals = df.loc[df[key_col] == key_value, count_col]
    return int(vals.iloc[0]) if len(vals) else 0


# Q1 counts + percentages
q1_total_journeys = int(len(journey_progress))
q1_reached_journeys = int(journey_progress["reached_tour_end"].sum())
q1_abandoned_journeys = q1_total_journeys - q1_reached_journeys
q1_reached_journey_rate = round(q1_reached_journeys / q1_total_journeys * 100, 2)
q1_abandoned_journey_rate = round(100 - q1_reached_journey_rate, 2)

q1_reached_users = int(q1_user_status["any_reached_end"].sum())
q1_total_users = int(len(q1_user_status))
q1_abandoned_only_users = q1_total_users - q1_reached_users
q1_reached_user_rate = round(q1_reached_users / q1_total_users * 100, 2)
q1_abandoned_user_rate = round(100 - q1_reached_user_rate, 2)

# Q2 counts + percentages
q2_active_share = summary_pct_or_zero(q2_mode_dist, "listening_mode", "Active listening")
q2_passive_share = summary_pct_or_zero(q2_mode_dist, "listening_mode", "Passive play")
q2_active_journeys = summary_count_or_zero(q2_mode_dist, "listening_mode", "Active listening", "tour_journeys")
q2_passive_journeys = summary_count_or_zero(q2_mode_dist, "listening_mode", "Passive play", "tour_journeys")

q2_active_reach_end = summary_pct_or_zero(
    q2_completion, "listening_mode", "Active listening", "reach_end_rate_pct"
)
q2_passive_reach_end = summary_pct_or_zero(
    q2_completion, "listening_mode", "Passive play", "reach_end_rate_pct"
)

# Q3 counts + percentages
q3_strict_follow_count = summary_count_or_zero(order_summary, "follows_common_order", True)
q3_proxy_follow_count = summary_count_or_zero(order_summary_proxy, "follows_common_order_proxy", True)
q3_strict_jump_count = summary_count_or_zero(jump_summary, "has_jump_event", True)
q3_proxy_jump_count = summary_count_or_zero(jump_summary_proxy, "has_jump_event_proxy", True)

q3_strict_follow = summary_pct_or_zero(order_summary, "follows_common_order", True)
q3_proxy_follow = summary_pct_or_zero(order_summary_proxy, "follows_common_order_proxy", True)
q3_strict_jump = summary_pct_or_zero(jump_summary, "has_jump_event", True)
q3_proxy_jump = summary_pct_or_zero(jump_summary_proxy, "has_jump_event_proxy", True)

summary = pd.DataFrame(
    {
        "metric": [
            "Q1 - Journeys reached tour end (count)",
            "Q1 - Journeys reached tour end (%)",
            "Q1 - Journeys abandoned before end (count)",
            "Q1 - Journeys abandoned before end (%)",
            "Q1 - Users reached end at least once (count)",
            "Q1 - Users reached end at least once (%)",
            "Q1 - Users only abandoned (count)",
            "Q1 - Users only abandoned (%)",
            "Q2 - Active journeys (count)",
            "Q2 - Active journey share (%)",
            "Q2 - Passive journeys (count)",
            "Q2 - Passive journey share (%)",
            "Q2 - Users with >=1 active journey (count)",
            "Q2 - Users with >=1 active journey (%)",
            "Q2 - Users passive-only (count)",
            "Q2 - Users passive-only (%)",
            "Q2 - Reached-end rate (active journeys, %)",
            "Q2 - Reached-end rate (passive journeys, %)",
            "Q3 strict - Follow common order (sessions, count)",
            "Q3 strict - Follow common order (%)",
            "Q3 strict - Has jump event (sessions, count)",
            "Q3 strict - Has jump event (%)",
            "Q3 proxy - Follow common order (sessions, count)",
            "Q3 proxy - Follow common order (%)",
            "Q3 proxy - Has jump event (sessions, count)",
            "Q3 proxy - Has jump event (%)",
            "Q3 strict - Sessions analyzed",
            "Q3 proxy - Sessions analyzed",
        ],
        "value": [
            q1_reached_journeys,
            q1_reached_journey_rate,
            q1_abandoned_journeys,
            q1_abandoned_journey_rate,
            q1_reached_users,
            q1_reached_user_rate,
            q1_abandoned_only_users,
            q1_abandoned_user_rate,
            q2_active_journeys,
            q2_active_share,
            q2_passive_journeys,
            q2_passive_share,
            q2_active_users,
            round(q2_active_users / q2_total_users * 100, 2),
            q2_passive_only_users,
            round(q2_passive_only_users / q2_total_users * 100, 2),
            q2_active_reach_end,
            q2_passive_reach_end,
            q3_strict_follow_count,
            q3_strict_follow,
            q3_strict_jump_count,
            q3_strict_jump,
            q3_proxy_follow_count,
            q3_proxy_follow,
            q3_proxy_jump_count,
            q3_proxy_jump,
            int(len(session_sequences)),
            int(len(session_sequences_proxy)),
        ],
    }
)

summary



**Το παρακατω κομματι κωδικα ειναι καθαρα για ελεγχo. δεν συμμετεχει στα ερωτηματα!!!!**

In [ ]:
import pandas as pd


dfs = {
    "events_clean": events_clean,
    # "bookings": bookings,   
}

# στήλες που θέλουμε να ελέγξουμε
cols_to_check = [
    "event_name",
    "platform",
    "language",
    "status",
    "booking_status",
    "product_type",
    "channel",
]

out_path = "unique_status_report.txt"

with open(out_path, "w", encoding="utf-8") as f:
    for df_name, df in dfs.items():
        f.write(f"\n=== {df_name} ===\n")
        for col in cols_to_check:
            if col in df.columns:
                f.write(f"\n-- {col} --\n")
                vc = df[col].astype("string").fillna("<NA>").value_counts(dropna=False)
                f.write(vc.to_string())
                f.write("\n")

print("Saved:", out_path)


## Παράρτημα Β: Ορισμοί Μεθοδολογίας και Παραδοχές

### 1. Πεδίο Ανάλυσης και Στόχος
Η ανάλυση απαντά στα 3 βασικά ερωτήματα:
1. Πόσο βαθιά καταναλώνουν οι χρήστες το περιεχόμενο ενός tour (Q1).
2. Αν η ακρόαση είναι ενεργητική ή παθητική (Q2).
3. Αν οι χρήστες ακολουθούν τη σειρά των stories ή κάνουν jump (Q3).

Όλοι οι δείκτες υπολογίζονται στα καθαρισμένα δεδομένα Ιουλίου-Οκτωβρίου 2025.

### 2. Σημειώσεις Δεδομένων και Περιορισμοί Καταγραφής
- Στο iOS το event `story_start` δεν καταγράφεται πάντα με συνέπεια.
- Για αυτό το Q3 δίνεται σε 2 εκδοχές:
  - **Strict**: Android + `story_start`.
  - **Proxy (cross-platform)**: Android + iOS με story-level proxy events.
- Στα mapping αρχεία δεν υπάρχει επίσημο πεδίο διάρκειας story.

### 3. Ορισμός Journey (`journey_idx`)
Ένα **journey** ορίζεται από το κλειδί:
- `user_key + tour_id + journey_idx`

Το `journey_idx` δημιουργείται με κανόνα χρονικού κενού:
- Ταξινομούμε τα events ανά `event_timestamp` μέσα σε κάθε `user_key + tour_id`.
- Αν το κενό μεταξύ 2 διαδοχικών events είναι **πάνω από 30 λεπτά**, ξεκινά νέο journey.

Σημαντικό:
- Το είδος event (π.χ. `story_listened_20/40/60/80`) δεν ανοίγει μόνο του νέο journey.
- Μόνο το χρονικό κενό επηρεάζει το split.

Παράδειγμα:
- Αν `story_listened_40` και `story_listened_60` απέχουν 35 λεπτά, ανήκουν σε διαφορετικά journeys.

### 4. Πότε Τελειώνει Ένα Journey
Ένα journey τελειώνει στο τελευταίο event πριν:
- από κενό >30 λεπτών, ή
- από το τέλος των διαθέσιμων δεδομένων για το συγκεκριμένο user-tour stream.

### 5. Πώς Ορίζεται το "Τέλος Tour" στο Q1
Το "τέλος tour" αξιολογείται ανά journey με canonical rank λογική:
1. Υπολογίζεται **canonical σειρά stories** ανά tour από median θέσεις παρατήρησης.
2. Βρίσκεται το `final_canonical_rank` (εκτιμώμενο τελευταίο rank story του tour).
3. Υπολογίζονται:
   - `reached_last_story`: μέγιστο rank που είδαμε >= `final_canonical_rank`
   - `reached_tour_end`: μέγιστο **end-like** rank >= `final_canonical_rank`

Το `end-like` rank μετρά μόνο stories με ισχυρή ένδειξη κατανάλωσης (depth >=80%).

Άρα στο Q1:
- `Reached tour end` = ισχυρή ένδειξη ότι έφτασε πραγματικά στο τέλος.
- `Abandoned before end` = δεν υπάρχει τέτοια ένδειξη.

Σε επίπεδο χρήστη:
- Χρήστης μετριέται ως "reached end" αν το πέτυχε σε **τουλάχιστον ένα** journey.

### 6. Ορισμοί Active / Passive (Q2)
Strong control events:
- `forward_10`, `backward_10`, `next_story`, `previous_story`, `click_progress_bar`

Ανά journey:
- `strong_control_events`: πλήθος strong controls.
- `controls_per_story = strong_control_events / stories_touched` (0 αν δεν υπάρχουν stories).

Κανόνας ταξινόμησης journey:
- **Active journey** αν:
  - `strong_control_events >= 2`, ή
  - `controls_per_story >= 0.2`
- **Passive journey** σε κάθε άλλη περίπτωση.

Γιατί `>=2` και όχι `>=1`:
- Μειώνει θόρυβο από τυχαίο/μεμονωμένο tap.
- Κρατά ένδειξη επαναλαμβανόμενης, συνειδητής αλληλεπίδρασης.

User-level labels:
- **Active only**: έχει active journeys, χωρίς passive.
- **Passive only**: δεν έχει κανένα active journey.
- **Mixed**: έχει και active και passive journeys.

### 7. Q3: Σειρά Stories vs Jumping
Για το order-following:
- Υπολογίζουμε `follows_common_order` (strict) ή `follows_common_order_proxy` (cross-platform).
- Βασίζεται σε μονοτονική πρόοδο των canonical ranks.

Για jump behavior:
- `has_jump_event` / `has_jump_event_proxy` είναι true όταν υπάρχει:
  - `previous_story`, `next_story`, `click_story`, `tour_item_clicked`

Σημαντική ερμηνεία:
- Το true/false του `follows_common_order` αθροίζει 100% μέσα στο δικό του metric.
- Το true/false του `has_jump_event` αθροίζει 100% μέσα στο δικό του metric.
- **Δεν αθροίζουμε** `follows_common_order % + has_jump_event %` μεταξύ τους, γιατί είναι διαφορετικές διαστάσεις συμπεριφοράς.

### 8. Time Metrics από `event_timestamp`
Με timestamps μπορούμε να υπολογίσουμε συμπεριφορικό χρόνο, αλλά όχι τέλεια πραγματική διάρκεια audio αρχείου.

Χρήσιμες έννοιες:
- **Elapsed time**: `max(timestamp) - min(timestamp)` σε story session ή journey.
- **Engaged time (proxy)**: άθροισμα event-to-event deltas με cap (π.χ. <=180 sec), ώστε να μειώνεται το idle/background inflation.

Επειδή δεν υπάρχει επίσημο πεδίο διάρκειας story, οι χρόνοι αντιμετωπίζονται ως behavioral proxies.

### 9. Τι Μπορεί να Αλλάξει τα Αποτελέσματα (Sensitivity)
Κύριοι μοχλοί ευαισθησίας:
- threshold split journey (30 vs 45/60 λεπτά),
- thresholds active ταξινόμησης (`>=2`, `0.2 controls/story`),
- threshold end-like evidence (σήμερα >=80%),
- strict vs proxy scope στο Q3.

Πρακτική σύσταση:
- Κρατάμε ένα baseline definition για τα επίσημα αποτελέσματα.
- Δίνουμε sensitivity checks στο παράρτημα για διαφάνεια και αξιοπιστία.



SOS 

Στα δεδομένα iOS παρατηρείται ασυνεπής καταγραφή του event story_start, σε αντίθεση με το Android όπου η κάλυψη είναι σαφώς πιο πλήρης. Για να διατηρηθεί η αξιοπιστία της ανάλυσης, το Q3 παρουσιάζεται σε δύο επίπεδα: (α) strict ανάλυση μόνο για Android με βάση το story_start και (β) proxy cross-platform ανάλυση για Android+iOS με story-level events που περιλαμβάνουν story_id. Ως πρόταση βελτίωσης, προτείνεται ενοποίηση του instrumentation στο iOS ώστε το story_start να καταγράφεται συστηματικά σε όλες τις εκδόσεις της εφαρμογής, μαζί με τακτικό QA έλεγχο πληρότητας events.

In [ ]:
# cell 21 - Final Totals & CORRECTED Durations
import pandas as pd
import numpy as np

# 1. Βασικά Στοιχεία
unique_users = analysis_events["user_key"].nunique()
total_journeys = len(journey_progress)

# 2. Υπολογισμός Διάρκειας Story (ΜΟΝΟ εντός του ίδιου Journey)
# Χρησιμοποιούμε το q12_events που έχει ήδη το journey_idx
story_stats = (
    q12_events.dropna(subset=["story_id"])
    .groupby(["user_key", "tour_id", "journey_idx", "story_id"])["event_timestamp"]
    .agg(start="min", end="max")
)
story_stats["duration_min"] = (story_stats["end"] - story_stats["start"]).dt.total_seconds() / 60

# Φιλτράρισμα: Κρατάμε μόνο διάρκειες > 0 και < 60 λεπτά (για να φύγουν τα λάθη)
valid_stories = story_stats[(story_stats["duration_min"] > 0) & (story_stats["duration_min"] < 60)]
mean_story_duration = valid_stories["duration_min"].mean()

# 3. Υπολογισμός Διάρκειας Tour (Journey)
# Χρησιμοποιούμε το q12_events για να δούμε τη συνολική διάρκεια κάθε journey
journey_times = (
    q12_events.groupby(["user_key", "tour_id", "journey_idx"])["event_timestamp"]
    .agg(start="min", end="max")
)
journey_times["duration_min"] = (journey_times["end"] - journey_times["start"]).dt.total_seconds() / 60

# Φιλτράρισμα: Ένα tour session λογικά διαρκεί από 2 λεπτά έως 4 ώρες (240 λεπτά)
valid_journeys = journey_times[(journey_times["duration_min"] > 1) & (journey_times["duration_min"] < 240)]
mean_tour_duration = valid_journeys["duration_min"].mean()

# --- ΕΚΤΥΠΩΣΗ ΑΠΟΤΕΛΕΣΜΑΤΩΝ ---

print("-" * 45)
print(f"📊 ΔΙΟΡΘΩΜΕΝΗ ΣΥΝΟΨΗ (Ιουλ - Οκτ 2025)")
print("-" * 45)
print(f"👤 Μοναδικοί Χρήστες: {unique_users:,}")
print(f"🚀 Συνολικά Journeys: {total_journeys:,}")
print(f"📈 Αναλογία Journeys ανά Χρήστη: {total_journeys/unique_users:.2f}")
print("-" * 45)
print(f"⏱️ Μέση Διάρκεια Story (Engagement): {mean_story_duration:.2f} λεπτά")
print(f"🗺️ Μέση Διάρκεια Tour (Engagement):  {mean_tour_duration:.2f} λεπτά")
print("-" * 45)

In [ ]:
# cell 22 - Memory Efficient Data Quality Audit
import pandas as pd
import numpy as np

# 1. Ορίζουμε μόνο τις στήλες που χρειαζόμαστε για τον έλεγχο
needed_cols = ['user_pseudo_id', 'event_timestamp', 'event_name', 'platform', 'event_date']

# 2. Ορίζουμε τύπους δεδομένων για εξοικονόμηση μνήμης
dtype_dict = {
    'platform': 'category',
    'event_name': 'category',
    'user_pseudo_id': 'string'
}

print("Φόρτωση δεδομένων (optimized)...")

# Διαβάζουμε το αρχείο επιλεκτικά
df_audit = pd.read_csv(
    OUTPUT_FILE, 
    usecols=needed_cols, 
    dtype=dtype_dict,
    low_memory=True # Το αφήνουμε True για καλύτερη διαχείριση μνήμης
)

# Μετατροπή timestamp (μόνο αφού φορτωθούν τα δεδομένα)
df_audit['event_timestamp'] = pd.to_datetime(df_audit['event_timestamp'])

# --- 1. Βασικά Descriptives ---
print("\n=== 1. ΒΑΣΙΚΑ DESCRIPTIVES ===")
print(f"Συνολικά Events: {len(df_audit):,}")
print(f"Μοναδικοί Χρήστες (Pseudo ID): {df_audit['user_pseudo_id'].nunique():,}")
print(f"Κατανομή Πλατφόρμας:\n{df_audit['platform'].value_counts()}")

# --- 2. Android Audit (Impossible Speed) ---
print("\n=== 2. ΕΛΕΓΧΟΣ ΑΚΕΡΑΙΟΤΗΤΑΣ (ANDROID BURSTS) ===")

# Κρατάμε μόνο Android για να ελαφρύνουμε τη μνήμη
android_check = df_audit[df_audit['platform'] == 'ANDROID'].copy()
del df_audit # Διαγράφουμε το αρχικό df για να ελευθερώσουμε RAM

# Σορτάρισμα για να δούμε τη σειρά των events ανά χρήστη
android_check = android_check.sort_values(by=['user_pseudo_id', 'event_timestamp'])

# Υπολογισμός διαφοράς χρόνου σε milliseconds
android_check['diff_ms'] = android_check.groupby('user_pseudo_id')['event_timestamp'].diff().dt.total_seconds() * 1000

# Εντοπισμός events < 10ms (Ύποπτα bursts)
fast_events = android_check[android_check['diff_ms'] <= 10].copy()

impossible_bursts = len(fast_events)
percent_of_total = (impossible_bursts / len(android_check)) * 100

print(f"Android Events: {len(android_check):,}")
print(f"Burst Events (<10ms): {impossible_bursts:,} ({percent_of_total:.2f}%)")

if impossible_bursts > 0:
    print("\nΣυχνότερα events σε bursts:")
    print(fast_events['event_name'].value_counts().head(10))
    
    print("\nΔείγμα ύποπτων bursts (πρώτες 10 γραμμές):")
    print(fast_events[['user_pseudo_id', 'event_name', 'event_timestamp', 'diff_ms']].head(10))

# Καθαρισμός μνήμης στο τέλος
del android_check
del fast_events

In [ ]:
# cell 23 - Deep Dive: Missing IDs, Platform Orphans & Case Issues
import pandas as pd
import numpy as np

# Φόρτωση δεδομένων (Memory Safe)
cols_to_audit = ['user_pseudo_id', 'tour_id', 'story_id', 'event_name', 'platform', 'event_timestamp']
df_gap = pd.read_csv(OUTPUT_FILE, usecols=cols_to_audit, low_memory=True)
df_gap['event_lower'] = df_gap['event_name'].str.lower()

print("=== 🔍 DEEP DIVE ΔΙΑΓΝΩΣΤΙΚΟΣ ΕΛΕΓΧΟΣ ===")

# --- 1. ΑΝΑΛΥΣΗ MISSING STORY_ID ---
print("\n1. ΓΙΑΤΙ ΛΕΙΠΕΙ ΤΟ STORY_ID (20.41%);")
# Βρίσκουμε ποια events δεν έχουν story_id
missing_story_events = df_gap[df_gap['story_id'].isna()]['event_name'].value_counts().head(10)
print("Top 10 Events χωρίς story_id:")
print(missing_story_events)
print("\n> Ερμηνεία: Αν εδώ βλέπεις events όπως 'tour_start' ή 'screen_view', είναι λογικό.")
print("> Αν βλέπεις 'story_listened_...', τότε υπάρχει σοβαρό θέμα στο instrumentation.")

# --- 2. ORPHAN EVENTS ΑΝΑ ΠΛΑΤΦΟΡΜΑ ---
print("\n2. ΟΡΦΑΝΑ EVENTS (COMPLETED ΧΩΡΙΣ START) ΑΝΑ ΠΛΑΤΦΟΡΜΑ:")

# Βρίσκουμε τα μοναδικά ζεύγη (user, story, platform) για starts και completes
starts = df_gap[df_gap['event_lower'] == 'story_start'][['user_pseudo_id', 'story_id', 'platform']].drop_duplicates()
completes = df_gap[df_gap['event_lower'] == 'story_completed'][['user_pseudo_id', 'story_id', 'platform']].drop_duplicates()

# Ταυτοποίηση ορφανών μέσω merge (left join)
# Ψάχνουμε completes που δεν έχουν αντίστοιχο start
orphans_df = completes.merge(starts, on=['user_pseudo_id', 'story_id'], how='left', suffixes=('_comp', '_start'))
orphans_only = orphans_df[orphans_df['platform_start'].isna()]

orphan_platform_counts = orphans_only['platform_comp'].value_counts()
print(f"Συνολικά ορφανά ζεύγη: {len(orphans_only):,}")
print("Κατανομή ανά Πλατφόρμα:")
for plat, count in orphan_platform_counts.items():
    print(f"- {plat}: {count:,} ορφανά")

# --- 3. ΑΝΑΦΟΡΑ ΔΙΠΛΟΕΓΓΡΑΦΩΝ CASE SENSITIVITY ---
print("\n3. ΣΥΓΚΕΚΡΙΜΕΝΑ EVENTS ΜΕ ΔΙΠΛΟΕΓΓΡΑΦΕΣ (CASE ISSUES):")
# Βρίσκουμε ποια event_names "συγκρούονται" όταν γίνονται πεζά
case_check = df_gap.groupby('event_lower')['event_name'].nunique()
conflicting_events = case_check[case_check > 1].index

if not conflicting_events.empty:
    for ev_lower in conflicting_events:
        variations = df_gap[df_gap['event_lower'] == ev_lower]['event_name'].unique()
        print(f"- Το event '{ev_lower}' εμφανίζεται ως: {list(variations)}")
else:
    print("- Δεν βρέθηκαν συγκρούσεις ονομάτων.")

# Καθαρισμός μνήμης
del df_gap

In [ ]:
# cell 24 - Ghost User Detection (Standalone)
import pandas as pd

print("=== 🕵️ ΕΝΤΟΠΙΣΜΟΣ GHOST USERS (OUTLIERS) ===")

# Διαβάζουμε ΜΟΝΟ το pseudo_id για να μη γεμίσει η RAM
# Χρησιμοποιούμε το OUTPUT_FILE που ορίσαμε στην αρχή
df_ghost = pd.read_csv(OUTPUT_FILE, usecols=['user_pseudo_id'], low_memory=True)

# Υπολογισμός events ανά χρήστη
user_activity = df_ghost['user_pseudo_id'].value_counts()

print("\nTop 10 πιο ενεργοί χρήστες (Events count):")
print(user_activity.head(10))

# Ορισμός ορίου: Πάνω από 2.000 events θεωρείται "ύποπτο"
# (Ένας κανονικός χρήστης σε ένα tour 1 ώρας παράγει συνήθως 50-150 events)
suspicious_threshold = 2000
suspicious_users = user_activity[user_activity > suspicious_threshold]

print(f"\nΣυνολικοί χρήστες: {len(user_activity):,}")
print(f"Χρήστες με πάνω από {suspicious_threshold} events: {len(suspicious_users)}")

if len(suspicious_users) > 0:
    print(f"\n ΠΡΟΣΟΧΗ: Ο κορυφαίος χρήστης έχει {user_activity.max(): } events.")
    print("Αν αυτό το νούμερο είναι εξωπραγματικό (π.χ. >10.000),")
    print("πρόκειται για bot, developer ή admin που βομβάρδισε το σύστημα με tests.")

# Καθαρίζουμε πάλι για ασφάλεια
del df_ghost